In [1]:
import lancedb
import openai
from dotenv import dotenv_values
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from motor.motor_asyncio import AsyncIOMotorClient

env = dotenv_values()
openai.api_key = env["OPENAI_API_KEY"]

In [2]:
mongo_client = AsyncIOMotorClient(env["OLD_MONGO_DSN"])
db = mongo_client["ontologies"]
await db["anatomic_locations"].count_documents({})

2901

In [10]:
coll = db["anatomic_locations"]
samples = await coll.find({}).to_list(length=2901)

In [13]:
vector_lengths = [len(s.get("embedding_vector")) for s in samples]
all(l == 1536 for l in vector_lengths)

True

In [2]:
db_conn = lancedb.connect(uri=env["LANCEDB_URI"], api_key=env["LANCEDB_API_KEY"])

In [3]:
db_conn.table_names()

['anatomic_locations', 'radlex', 'snomedct']

In [4]:
embedding_function = get_registry().get("openai").create(name="text-embedding-3-large", dim=1536)

In [5]:
class ConceptIndexNode(LanceModel):
    concept_id: str
    concept_text: str = embedding_function.SourceField()
    vector: Vector(dim=embedding_function.ndims()) = embedding_function.VectorField()


In [26]:
samples[0].keys()

dict_keys(['_id', 'acrCommonId', 'snomedId', 'snomedDisplay', 'description', 'region', 'containedByRef', 'containsRefs', 'synonyms', 'hasPartsRefs', 'codes', 'definition', 'embedding_vector'])

In [41]:
from typing import Any


def get_data_from_node(node: dict[str, Any]) -> dict[str, Any]:
    text = node["description"]
    if node.get("definition"):
        text += f"\nDefinition: {node['definition']}"
    if node.get("synonyms"):
        text += f" (synonyms: {'; '.join(node['synonyms'])})"
    return {"concept_id": node["_id"], "concept_text": text, "vector": node["embedding_vector"]}

In [42]:
for sample in samples[:50]:
    data = get_data_from_node(sample)
    print(data["concept_id"])
    print(data["concept_text"])
    print()

RID56
abdomen
Definition: Subdivision of trunk proper which is demarcated from the thorax internally by the inferior surface of the sternocostal part of the diaphragm and externally by the costal margin, from the back of abdomen by the external surface of the posterior abdominal wall,  from the perineum by the superior surface of the urogenital diaphragm and from the lower limbs by the inguinal folds; together with the thorax, and perineum, it constitutes the trunk proper. Examples: There is only one abdomen. [FMA] (synonyms: abdominopelvis; abdominopelvic region)

RID905
abdominal aorta
Definition: The aorta from the diaphragm to the bifurcation into the right and left common iliac arteries. [MeSH]

RID294
uterine adnexa
Definition: Appendages of the uterus which include the fallopian tubes, the ovary, and the supporting ligaments of the uterus (broad ligament; round ligament). [MeSH] (synonyms: adnexa)

RID294_RID5824
left uterine adnexa

RID294_RID5825
right uterine adnexa

RID935
a

In [43]:
data_to_load = [get_data_from_node(sample) for sample in samples]

In [44]:
len(data_to_load)

2901

In [48]:
len(data_to_load[0]["vector"])

1536

In [49]:
db_conn.create_table("anatomic_locations", data=data_to_load, schema=ConceptIndexNode)

RemoteTable(cdetools-w5qh9c.anatomic_locations)

In [6]:
db_conn.table_names()

['anatomic_locations']

In [6]:
anlocs = db_conn.open_table("anatomic_locations")

In [7]:
anlocs.count_rows()

2901

In [10]:
anlocs.search("heart").to_list()

[{'concept_id': 'RID1401',
  'concept_text': 'chordae tendineae\nDefinition: The tendinous cords that connect each cusp of the two atrioventricular heart valves to appropriate papillary muscles in the heart ventricles, preventing the valves from reversing themselves when the ventricles contract. [MeSH] (synonyms: tendinous cords of heart)',
  'vector': [0.009492023847997189,
   0.040259987115859985,
   -0.022601371631026268,
   0.028331100940704346,
   0.007220841012895107,
   0.05340385064482689,
   -0.01423458382487297,
   -0.018638882786035538,
   -0.024410033598542213,
   -0.006709997542202473,
   0.0037070666439831257,
   0.06190870329737663,
   -0.006109411362558603,
   -0.005595116410404444,
   -0.037001632153987885,
   -0.019066886976361275,
   -0.03437838330864906,
   0.03291488438844681,
   0.03388134762644768,
   -0.024092482402920723,
   -0.0297945998609066,
   -0.021275939419865608,
   -0.00327733694575727,
   -0.04542364552617073,
   0.034571677446365356,
   0.01932921074

In [ ]:
{(r["concept_id"], r["concept_text"], r["_score"]) for r in Out[10]}

{('RID115',
  'gastric cardia\nDefinition: That part of the stomach close to the opening from esophagus into the stomach (cardiac orifice), the esophagogastric junction. The cardia is so named because of its closeness to the heart. Cardia is characterized by the lack of acid-forming cells (gastric parietal cells). [MeSH] (synonyms: cardia; cardial part of stomach; pars cardiaca gastricae)',
  1.91997492313385),
 ('RID1231',
  'pulmonary vein\nDefinition: The veins that return the oxygenated blood from the lungs to the left atrium of the heart. [MeSH]',
  3.9288830757141113),
 ('RID1385',
  'heart\nDefinition: The hollow, muscular organ that maintains the circulation of the blood. [MeSH]',
  4.932381629943848),
 ('RID1395',
  'mitral valve\nDefinition: The valve between the left atrium and left ventricle of the heart. [MeSH]',
  4.596917629241943),
 ('RID1397',
  'tricuspid valve\nDefinition: The valve consisting of three cusps situated between the right atrium and right ventricle of th

In [9]:
anlocs.create_fts_index("concept_text")

In [ ]:
from lancedb.rerankers import OpenaiReranker


In [9]:
# results = anlocs.search("pcl tear", query_type="hybrid", vector_column_name="vector").rerank(reranker=reranker).to_list()
results = anlocs.search("pcl tear", query_type="hybrid", vector_column_name="vector").to_list()

[(r["concept_id"], r["concept_text"], r["_relevance_score"]) for r in results]

[('RID2784',
  'posterior cruciate ligament\nDefinition: A strong ligament of the knee that originates from the anterolateral surface of the medial condyle of the femur, passes posteriorly and inferiorly between the condyles, and attaches to the posterior intercondylar area of the tibia. [MeSH] (synonyms: ligamentum cruciatum posterius)',
  0.016393441706895828),
 ('RID48974', 'left posterior cruciate ligament', 0.016129031777381897),
 ('RID48973', 'right posterior cruciate ligament', 0.01587301678955555),
 ('RID48531', 'left patella', 0.015625),
 ('RID39500', 'levator ani', 0.015384615398943424),
 ('RID48972', 'left anterior cruciate ligament', 0.01515151560306549),
 ('RID48971', 'right anterior cruciate ligament', 0.014925372786819935),
 ('RID3016', 'posterior tibiotalar ligament', 0.014705882407724857),
 ('RID48530', 'right patella', 0.014492753893136978),
 ('RID2781',
  'anterior cruciate ligament\nDefinition: A strong ligament of the knee that originates from the posteromedial por

In [10]:
results[0].keys()

dict_keys(['concept_id', 'concept_text', 'vector', '_relevance_score'])

In [11]:
anlocs.embedding_functions

{'vector': EmbeddingFunctionConfig(vector_column='vector', source_column='concept_text', function=OpenAIEmbeddings(max_retries=7, name='text-embedding-3-large', dim=1536, base_url=None, default_headers=None, organization=None, api_key=None, use_azure=False))}

In [12]:
await db["radlex"].count_documents({})

NameError: name 'db' is not defined

In [51]:
radlex_docs = await db["radlex"].find({}).to_list()

In [53]:
all([len(d["embedding_vector"]) == 1536 for d in radlex_docs])

True

In [54]:
radlex_docs[0].keys()

dict_keys(['_id', 'preferredLabel', 'parent', 'radlexProperties', 'embedding_vector'])

In [ ]:
from typing import Any


def get_data_from_radlex_node(node: dict[str, Any]) -> dict[str, Any]:
    text = node["preferredLabel"]
    if node.get("definition"):
        text += f"\n: {node['definition']}"
    if node.get("synonyms"):
        text += f" (synonyms: {'; '.join(node['synonyms'])})"
    return {"concept_id": node["_id"], "concept_text": text, "vector": node["embedding_vector"]}

In [60]:
for sample in radlex_docs[:20]:
    data = get_data_from_radlex_node(sample)
    print(data["concept_id"])
    print(data["concept_text"])
    print(len(data["vector"]))
    print()

RID38411
postcentral branch of spinal branch of left third lumbar artery
1536

RID22326
trunk of superficial transverse perineal muscle branch of perineal nerve
1536

RID41389
tendon of left second interspinalis lumborum
1536

RID14646
lingual branch of left glossopharyngeal nerve to left postsulcal part of tongue
1536

RID46248
lateral pectoral nerve branch of anterior division of anterior ramus of left fifth cervical nerve
1536

RID33917
sulcal segment of gyrus of right dentate gyrus
1536

RID19482
neuronal component of L2 segment
1536

RID37008
internal pyramidal lamina of right Brodmann area 12
1536

RID2179
RID2179
1536

RID17673
left spinal trigeminal tract of pons
1536

RID35145
dilated transverse colon sign
: Dilated transverse colon with empty cecum and ascending colon on a frontal radiograph. Suggests appendicitis, often with perforated appendix.
1536

RID44880
suprasternal branch of suprascapular artery (synonyms: suprasternal part of suprascapular artery)
1536

RID5907
sept

In [62]:
from itertools import batched

In [63]:
data_to_load = [get_data_from_radlex_node(sample) for sample in radlex_docs]
len(data_to_load)

46761

In [64]:
def batched_data_to_load(data_to_load, batch_size=100):
    for batch in batched(data_to_load, batch_size):
        yield batch

In [ ]:
[len(b) for b in batched_data_to_load(data_to_load)]

In [71]:
db_conn.create_table("radlex", data=data_to_load, schema=ConceptIndexNode)

RemoteTable(cdetools-w5qh9c.radlex)

In [13]:
radlex = db_conn.open_table("radlex")

In [73]:
radlex.create_fts_index("concept_text")

In [14]:
results = (
    radlex.search("posterior cruciate ligament tear", query_type="hybrid", vector_column_name="vector")
    .rerank(reranker=reranker)
    .to_list()
)
# results = radlex.search("pcl tear", query_type="hybrid", vector_column_name="vector").to_list()

[(r["concept_id"], r["concept_text"], r["_relevance_score"]) for r in results]


[('RID2784',
  'posterior cruciate ligament\n: A strong ligament of the knee that originates from the anterolateral surface of the medial condyle of the femur, passes posteriorly and inferiorly between the condyles, and attaches to the posterior intercondylar area of the tibia. [MeSH] (synonyms: ligamentum cruciatum posterius; Hinteres Kreuzband)',
  1.0),
 ('RID48250',
  'posteromedial band of left posterior cruciate ligament',
  0.8999999761581421),
 ('RID48249',
  'posteromedial band of right posterior cruciate ligament',
  0.8999999761581421),
 ('RID48246',
  'anterolateral band of right posterior cruciate ligament',
  0.8999999761581421),
 ('RID48247',
  'anterolateral band of left posterior cruciate ligament',
  0.8999999761581421),
 ('RID48245',
  'anterolateral band of posterior cruciate ligament (synonyms: anterolateral bundle of posterior cruciate ligament)',
  0.8999999761581421),
 ('RID48248',
  'posteromedial band of posterior cruciate ligament (synonyms: posteromedial bun

In [8]:
snomedct = db_conn.open_table("snomedct")

In [9]:
snomedct.count_rows()

508540

In [16]:
snomedct.list_indices()

[Index(IvfPq, columns=["vector"], name="vector_idx"),
 Index(IvfPq, columns=["vector"], name="vector_idx"),
 Index(IvfPq, columns=["vector"], name="vector_idx"),
 Index(FTS, columns=["concept_text"], name="concept_text_idx"),
 Index(Bitmap, columns=["concept_id"], name="concept_id_idx")]

In [13]:
snomedct.create_scalar_index(column="concept_id", index_type="BITMAP")

In [15]:
anlocs = db_conn.open_table("anatomic_locations")
anlocs.count_rows()

2901

In [19]:
anlocs.create_scalar_index(column="concept_id", index_type="BITMAP")

In [17]:
radlex = db_conn.open_table("radlex")
radlex.count_rows()


46761

In [18]:
radlex.create_scalar_index(column="concept_id", index_type="BITMAP")